In [ ]:
!pip install -q \
langchain \
langchain-community \
langchain-core \
langchain-text-splitters \
langchain-pinecone \
langchain-groq \
langchain-huggingface \
sentence-transformers \
transformers \
pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 71.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 101.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.6/587.6 kB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.3/259.3 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 8.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not curren

In [ ]:
import os
PINECONE_API_KEY = PINECONE_API_KEY
GROQ_API_KEY     = GROQ_API_KEY
NGROK_AUTH_TOKEN = NGROK_AUTH_TOKEN

In [ ]:
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
print('✅ API keys set!')

✅ API keys set!


In [ ]:
from google.colab import files
os.makedirs("/content/data",exist_ok=True)
print('📂 Upload your medical PDF files now...')
uploaded = files.upload()

📂 Upload your medical PDF files now...


Saving Medical_book.pdf to Medical_book.pdf


In [ ]:
for filename in uploaded.keys():
  os.rename(filename,f"/content/data/{filename}")
  print(f'  ✅ Moved: {filename} → /content/data/')
print(f'\n Total pdfs ready:{len(os.listdir("/content/data"))}')

  ✅ Moved: Medical_book.pdf → /content/data/

 Total pdfs ready:1


In [ ]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from typing import List

In [ ]:
def load_pdf_files(data):
  loader = DirectoryLoader(
      data,
      glob="*.pdf",
      loader_cls=PyPDFLoader
  )
  return loader.load()

In [ ]:
def filter_to_minimal_docs(docs:List[Document]) -> list[Document]:
  return [
      Document(
          page_content = doc.page_content,
          metadata={"source":doc.metadata.get("source")}
      )
      for doc in docs
  ]

In [ ]:
def text_split(docs):
  splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=20)
  return splitter.split_documents(docs)

In [ ]:
print("Loading pdfs")
extracted_data = load_pdf_files("/content/data")
minimal_docs = filter_to_minimal_docs(extracted_data)
text_chunks = text_split(minimal_docs)
print(f'Loaded {len(extracted_data)} pages {len(text_chunks)} chunks')

Loading pdfs
Loaded 637 pages 5860 chunks


In [ ]:
!pip install numpy==1.26.4

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
test_vec = embedding.embed_query("test")
print(f'✅ Embedding model ready! Vector size: {len(test_vec)}')

✅ Embedding model ready! Vector size: 384


In [ ]:
from pinecone import Pinecone,ServerlessSpec

creating index at pinecone

In [ ]:
index_name = "medical-chatbot"
pc = Pinecone(api_key=PINECONE_API_KEY)
#creating index if it doesnt exist
if not pc.has_index(index_name):
  pc.create_index(
      name = index_name,
      dimension=384,
      metric="cosine",
      spec=ServerlessSpec(cloud="aws", region="us-east-1")
  )
print("Index Created")

Index Created


In [ ]:
#uploading doc to pinecone
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore(
    index_name=index_name,
    embedding=embedding
)

docsearch.add_documents(text_chunks)

['fafb52bc-2b8a-4280-a3d0-fd8b5f8e62b0',
 '01e527fe-a6b3-4656-9227-8536c8494413',
 '638887ac-efc1-4ad3-8c80-d974cfb35c00',
 '19c24ff8-678a-4e04-ac4f-f964ca413172',
 '360de564-0648-4b90-8d64-1145014cc4ca',
 'd2c55a9f-9bcb-42c9-8c57-03b72d54b790',
 '4d74b3af-56d3-4f73-85ca-943a361a965b',
 '5df7eedc-4415-4324-95b6-d1f435386439',
 'a37a42e7-f626-4042-a416-e15da16048c8',
 '50804ea7-d35e-4c17-8a19-036faaca07c3',
 '8ec58f17-b520-4f40-886d-015dacf7e1cb',
 '33f0e0c7-a8c3-4daf-956b-4dc9604ceeec',
 'c1f462d5-b1e8-4ede-8c6d-feb8648bce07',
 '01251347-8195-411e-9f86-739203a79db8',
 '81ff42cb-e12a-4404-97c0-b55ad0e043e0',
 '5a2b1d02-f7fc-4545-981c-677a949b5f2d',
 'd970410c-98f7-4036-a2b9-630d75523c07',
 'debb1300-4bf0-4093-bf53-3a0b891547f0',
 'd67cb3d2-f836-43ee-aba8-ff08979b9d48',
 'a90973f6-dc88-4549-a6ec-5424d25abbc7',
 '96a2f13d-426a-4ab1-88b6-bccc9d38c412',
 '900e9d5d-f742-4093-a5b9-c84f1bef31c3',
 '4ab7c1ea-b4ec-4ec5-8c85-0222a4cdec45',
 'd609ded1-9b40-47b1-9966-d5356aa6d596',
 '2aad168c-bc8b-

In [ ]:
retriever = docsearch.as_retriever(search_type='similarity',search_kwargs={'k':3})

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
chatModel = ChatGroq(
    model_name="llama-3.1-8b-instant",
    api_key=GROQ_API_KEY,
    temperature=0.2
)

In [ ]:
system_prompt = (
    "You are a Medical assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)
prompt = ChatPromptTemplate.from_messages([
    ("system",system_prompt),
    ("human","{input}")
])

In [ ]:
question_answer_chain = (
    {"context": retriever, "input": lambda x: x}
    | prompt
    | chatModel
)

In [ ]:
response = question_answer_chain.invoke("hii")

In [ ]:
print(response.content)

Hello. How can I assist you today?


Flask API + Ngrok for connecting to vs code

In [ ]:
!pip install flask flask-cors pyngrok

In [ ]:
from pyngrok import ngrok
ngrok.kill()

In [ ]:
from pyngrok import ngrok

In [ ]:
from flask import Flask, request, jsonify
from flask_cors import CORS

In [ ]:
import threading

In [ ]:
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok, conf
import threading

# 🔐 Set ngrok token
conf.get_default().auth_token = NGROK_AUTH_TOKEN

app = Flask(__name__)
CORS(app)

@app.route("/chat", methods=["POST"])
def chat():
    try:
        data = request.get_json()
        question = data.get("question", "").strip()

        if not question:
            return jsonify({"error": "No question provided"}), 400

        # ✅ FIXED HERE
        result = question_answer_chain.invoke({"input": question})

        answer = getattr(result, "content", "I don't know.")

        if not answer.strip():
            answer = "I don't know."

        return jsonify({
            "answer": answer,
            "question": question
        })

    except Exception as e:
        print("ERROR:", str(e))
        return jsonify({
            "answer": "Something went wrong. Try again."
        }), 200


@app.route("/health", methods=["GET"])
def health():
    return jsonify({
        "status": "ok",
        "model": "llama-3.1-8b-instant"
    })

# Run Flask in background
def run_flask():
    app.run(port=5000, debug=False, use_reloader=False)

thread = threading.Thread(target=run_flask, daemon=True)
thread.start()

# Start ngrok
public_url = ngrok.connect(5000)

print("\n" + "="*60)
print("🚀 Medical Chatbot API is LIVE!")
print("="*60)
print(f"\n🌐 PUBLIC URL:")
print(f"   {public_url.public_url}")
print(f"\n📡 API Endpoints:")
print(f"   POST {public_url.public_url}/chat")
print(f"   GET  {public_url.public_url}/health")
print("="*60)

 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.



🚀 Medical Chatbot API is LIVE!

🌐 PUBLIC URL:
   https://friskily-spiriferous-irene.ngrok-free.dev

📡 API Endpoints:
   POST https://friskily-spiriferous-irene.ngrok-free.dev/chat
   GET  https://friskily-spiriferous-irene.ngrok-free.dev/health
